In [1]:
import os

In [2]:
pwd = os.getcwd()

In [3]:
os.chdir("../")
print(os.getcwd())

/Users/xenan.bilgin/Projects/MLops/mlops-project1-wine-quality


In [4]:
import pandas as pd

data = pd.read_csv("artifacts/data_ingestion/winequality-red.csv")

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [6]:
# Create schema.yaml file for data validation
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path

In [8]:
from src.mlops1_data_science_project import logger
from src.mlops1_data_science_project.constants import (
    CONFIG_FILE_PATH,
    PARAMS_FILE_PATH,
    SCHEMA_FILE_PATH,
)
from src.mlops1_data_science_project.utils.common import create_directories, read_yaml

In [10]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir, data_path=config.data_path
        )
        return data_transformation_config

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def train_test_split(
        self, data: pd.DataFrame, test_size: float = 0.2, random_state: int = 42
    ):
        train_data, test_data = train_test_split(
            data, test_size=test_size, random_state=random_state
        )
        output_dir = Path(self.config.root_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        train_data.to_csv(output_dir / "train_data.csv", index=False)
        test_data.to_csv(output_dir / "test_data.csv", index=False)
        logger.info(f"Train and test data saved at {output_dir}")
        logger.info(
            f"Train data shape: {train_data.shape}, Test data shape: {test_data.shape}"
        )

In [ ]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.train_test_split(data)
    logger.info("Data transformation completed successfully")
except Exception as e:
    logger.exception(e)

[2026-05-27 16:49:37,994: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-27 16:49:37,995: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-27 16:49:37,997: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-05-27 16:49:37,998: INFO: common: created directory at: artifacts]
[2026-05-27 16:49:37,998: INFO: common: created directory at: artifacts/data_validation]
[2026-05-27 16:49:38,000: INFO: 1253244569: All columns are valid]
[2026-05-27 16:49:38,001: INFO: 2729420827: Data validation status: True]
